# (노트) CAM - hynn, iu

- toc:true
- branch: master
- badges: true
- comments: true
- author: 신록예찬
- categories: [빅데이터분석]

### import

In [ ]:
from fastai.vision.all import * 
import torch 

In [ ]:
def search_images_ddg(key,max_n=500):
    """Search for 'key' with DuckDuckGo and return a unique urls of 'max_n' images
       (Adopted from https://github.com/deepanprabhu/duckduckgo-images-api)
    """
    url        = 'https://duckduckgo.com/'
    params     = {'q':key}
    res        = requests.post(url,data=params)
    searchObj  = re.search(r'vqd=([\d-]+)\&',res.text)
    if not searchObj: print('Token Parsing Failed !'); return
    requestUrl = url + 'i.js'
    headers    = {'User-Agent': 'Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:71.0) Gecko/20100101 Firefox/71.0'}
    params     = (('l','us-en'),('o','json'),('q',key),('vqd',searchObj.group(1)),('f',',,,'),('p','1'),('v7exp','a'))
    urls       = []
    while True:
        try:
            res  = requests.get(requestUrl,headers=headers,params=params)
            data = json.loads(res.text)
            for obj in data['results']:
                urls.append(obj['image'])
                max_n = max_n - 1
                if max_n < 1: return L(set(urls))     # dedupe
            if 'next' not in data: return L(set(urls))
            requestUrl = url + data['next']
        except:
            pass

In [ ]:
# class Hook(): ## 상태를 기록하는 클래스 
#     def hook_func(self,module, input, output): self.stored = output 

### data 

In [ ]:
keywords = 'dog', 'cat' 
path=Path('CAM')

In [ ]:
if not path.exists(): # 현재폴더에 CAM라는 폴더가 있는지 체크 
    path.mkdir() # 현재폴더에 CAM라는 폴더가 만들어짐 
    for keyword in keywords: # keyword='dog', keyword='wolf' 일때 아래내용을 반복 
        lastpath=path/keyword # ./CAM/dog or ./CAM/wolf 
        lastpath.mkdir(exist_ok=True) # make ./CAM/dog or ./CAM/wolf 
        urls=search_images_ddg(keyword) # 'dog','wolf' 검색어로 url들의 리스트를 얻음
        download_images(lastpath,urls=urls) # 그 url에 해당하는 이미지들을  ./CAM/dog or ./CAM/wolf 에 저장

In [ ]:
verify_images(get_image_files(path)) # 이상한 그림파일 리스트확인

In [ ]:
verify_images(get_image_files(path)).map(Path.unlink) # 이상한 그림파일 삭제

In [ ]:
verify_images(get_image_files(path)) # 다시 리스트확인 

In [ ]:
dls = ImageDataLoaders.from_folder(
    path,
    train='CAM',
    valid_pct=0.2,
    seed=43052,
    item_tfms=Resize(224))   

In [ ]:
dls.show_batch(max_n=16)

In [ ]:
path= untar_data(URLs.PETS)/'images'

In [ ]:
files=get_image_files(path) 

In [ ]:
files[2] # txt 파일의 3번째 목록

In [ ]:
def label_func(f):
    if f[0].isupper():
        return 'cat' 
    else: 
        return 'dog' 

In [ ]:
dls=ImageDataLoaders.from_name_func(path,files,label_func,item_tfms=Resize(224),valid_pct=0.2) 

### learn 

In [ ]:
# lrnr=cnn_learner(dls,resnet18,metrics=error_rate)
# lrnr.fine_tune(3)

In [ ]:
lrnr=cnn_learner(dls,resnet34,metrics=error_rate)
lrnr.fine_tune(1)

In [ ]:
# lrnr.show_results(max_n=16)

### CAM one_batch

`-` 개 혹은 늑대라고 판단한 근거를 알아보자. 

#### 하니

In [ ]:
x,  =first(dls.test_dl([PILImage.create('2021-08-25-Hani01.jpeg')]))
dls.vocab

In [ ]:
lrnr.model(x)

In [ ]:
cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
x_dec=TensorImage(dls.train.decode((x,))[0][0])

fig,((ax1,ax2),(ax3,ax4))= plt.subplots(2,2)
x_dec.show(ctx=ax1)

x_dec.show(ctx=ax2)
ax2.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),interpolation='bilinear',cmap='magma');
ax3.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),cmap='magma');
ax4.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),interpolation='bilinear',cmap='magma');

fig.set_figwidth(12)
fig.set_figheight(12)
fig.tight_layout()

#### 개

In [ ]:
img=PILImage.create(get_image_files(path)[0])
x,  = first(dls.test_dl([img]))
cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
x_dec=TensorImage(dls.train.decode((x,))[0][0])
fig,((ax1,ax2),(ax3,ax4))= plt.subplots(2,2)
x_dec.show(ctx=ax1)
x_dec.show(ctx=ax2)
ax2.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),interpolation='bilinear',cmap='magma');
ax3.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),cmap='magma');
ax4.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),interpolation='bilinear',cmap='magma');
fig.set_figwidth(12)
fig.set_figheight(12)
fig.tight_layout()

#### 고양이

In [ ]:
img=PILImage.create(get_image_files(path)[521]) # 31
x,  = first(dls.test_dl([img]))

cam_map= torch.einsum('ck,kij->cij',lrnr.model[1][-1].weight,lrnr.model[0](x)[0])
x_dec=TensorImage(dls.train.decode((x,))[0][0])

fig,((ax1,ax2),(ax3,ax4))= plt.subplots(2,2)
x_dec.show(ctx=ax1)
x_dec.show(ctx=ax2)
ax2.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),interpolation='bilinear',cmap='magma');
ax3.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),cmap='magma');
ax4.imshow(cam_map[0].detach().cpu(),alpha=0.8,extent=(0,224,224,0),interpolation='bilinear',cmap='magma');
fig.set_figwidth(12)
fig.set_figheight(12)
fig.tight_layout()

- 노란색: 활성이 큰 영역 
- 보라색: 활성이 낮은 영역 